In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

In [2]:
import pandas as pd
import numpy as np
import re
from matplotlib import pyplot as plt
#from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

import sys
sys.path.append('../../ss_lal_military/src')
sys.path.append('../src/')
sys.path.append('../migrant/notebooks/utilities/')

In [3]:
import os
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_agent_full_script', 4) # For example, MyPySpark3
  
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


Spark3 Starting
Kernel: mlpy3811v23, Python_path: /data/sdp/mlpy3811v23/bin/python, Resource_level: 4.middle+CPU(101,728)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/16 18:34:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/10/16 18:34:21 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/10/16 18:34:21 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/10/16 18:34:21 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/10/16 18:34:21 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/10/16 18:36:34 WARN HiveConf: HiveConf of name hive.mapred.supports.subdirectories does not exist
25/10/16 18:36:36 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
25/10/16 18:36:52 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


In [ ]:
spark.sql('''
select count(*) from arnsdpsbx_team_ss.bpm_cluster_premier_features_2025_04_30_last_new_v9
where (sum_is_call_s_dkm_3m is null or sum_is_call_s_dkm_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
and (sum_is_meet_km_3m is null or sum_is_meet_km_3m = 0)
and (sum_is_call_s_km_3m is null or sum_is_call_s_km_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
and count_sale_vsp_3m_new is not null
and count_sale_sbol_3m is not null
and count_sales_with_predlozh_3m is null
and sale_flag_erkc_3m is null
''').show()




spark.sql('''
select count(*) from arnsdpsbx_team_ss.bpm_cluster_premier_features_2025_04_30_last_new_v9
where count_sale_vsp_3m_new is null
and count_sale_sbol_3m is not null and (count_sales_with_predlozh_3m is null or sale_flag_erkc_3m is null or sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0
    or sum_is_call_s_km_3m is null or sum_is_call_s_km_3m = 0 or sum_is_meet_km_3m is null or sum_is_meet_km_3m = 0
    or sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0 or sum_is_call_s_dkm_3m is null or sum_is_call_s_dkm_3m = 0)
''').show()

# Train

In [14]:
target = spark.sql('''
with first_going_self as (
select epk_id, report_dt, '1' as target from arnsdpsbx_team_ss.bpm_cluster_premier_features_2025_04_30_last_new_v9
where count_sale_vsp_3m_new is not null and count_sale_sbol_3m is not null
and count_sales_with_predlozh_3m is null and sale_flag_erkc_3m is null
and (sum_is_call_s_dkm_3m is null or sum_is_call_s_dkm_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
and (sum_is_meet_km_3m is null or sum_is_meet_km_3m = 0)
and (sum_is_call_s_km_3m is null or sum_is_call_s_km_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
),

second_going_km as (
select epk_id, report_dt, '2' as target from arnsdpsbx_team_ss.bpm_cluster_premier_features_2025_04_30_last_new_v9
where count_sale_vsp_3m_new is not null and count_sale_sbol_3m is null
and (count_sales_with_predlozh_3m is not null or sale_flag_erkc_3m is not null
or (sum_is_call_s_dkm_3m is not null or sum_is_call_s_dkm_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0)
or (sum_is_meet_km_3m is not null or sum_is_meet_km_3m > 0)
or (sum_is_call_s_km_3m is not null or sum_is_call_s_km_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0))
),

third_not_going_self as (
select epk_id, report_dt, '3' as target from arnsdpsbx_team_ss.bpm_cluster_premier_features_2025_04_30_last_new_v9
where count_sale_vsp_3m_new is null and count_sale_sbol_3m is not null
and count_sales_with_predlozh_3m is null and sale_flag_erkc_3m is null
and (sum_is_call_s_dkm_3m is null or sum_is_call_s_dkm_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
and (sum_is_meet_km_3m is null or sum_is_meet_km_3m = 0)
and (sum_is_call_s_km_3m is null or sum_is_call_s_km_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
),

four_not_going_km as (
select epk_id, report_dt, '4' as target from arnsdpsbx_team_ss.bpm_cluster_premier_features_2025_04_30_last_new_v9
where count_sale_vsp_3m_new is null and count_sale_sbol_3m is null
and (count_sales_with_predlozh_3m is not null or sale_flag_erkc_3m is not null
or (sum_is_call_s_dkm_3m is not null or sum_is_call_s_dkm_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0)
or (sum_is_meet_km_3m is not null or sum_is_meet_km_3m > 0)
or (sum_is_call_s_km_3m is not null or sum_is_call_s_km_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0))
)

(select * from first_going_self)
union all
(select * from second_going_km)
union all
(select * from third_not_going_self)
union all
(select * from four_not_going_km)
''')

target.write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_target_2025_04_30_multiclass', mode='overwrite')

In [15]:
spark.sql('''
select count(epk_id), count(distinct epk_id) from arnsdpsbx_team_ss.bpm_premier_target_2025_04_30_multiclass
''').show()

+-------------+----------------------+
|count(epk_id)|count(DISTINCT epk_id)|
+-------------+----------------------+
|      1260975|               1260975|
+-------------+----------------------+



In [16]:
spark.sql('''
select target, count(epk_id) from arnsdpsbx_team_ss.bpm_premier_target_2025_04_30_multiclass
group by target
order by target
''').show()

+------+-------------+
|target|count(epk_id)|
+------+-------------+
|     1|       109304|
|     2|       114531|
|     3|       420211|
|     4|       616929|
+------+-------------+



In [7]:
spark.sql('''
select count(epk_id) from prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth
where report_dt = '2025-07-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS'
''').show()

+-------------+
|count(epk_id)|
+-------------+
|      3676225|
+-------------+



# OOT

In [6]:
target_oot = spark.sql('''
with first_going_self as (
select epk_id, report_dt, '1' as target from arnsdpsbx_team_ss.bpm_multiclass_premier_main_feature_target_2025_07_31
where count_sale_vsp_3m is not null and count_sale_sbol_3m is not null
and count_sales_with_predlozh_3m is null and sale_flag_erkc_3m is null
and (sum_is_call_s_dkm_3m is null or sum_is_call_s_dkm_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
and (sum_is_meet_km_3m is null or sum_is_meet_km_3m = 0)
and (sum_is_call_s_km_3m is null or sum_is_call_s_km_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
),

second_going_km as (
select epk_id, report_dt, '2' as target from arnsdpsbx_team_ss.bpm_multiclass_premier_main_feature_target_2025_07_31
where count_sale_vsp_3m is not null and count_sale_sbol_3m is null
and (count_sales_with_predlozh_3m is not null or sale_flag_erkc_3m is not null
or (sum_is_call_s_dkm_3m is not null or sum_is_call_s_dkm_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0)
or (sum_is_meet_km_3m is not null or sum_is_meet_km_3m > 0)
or (sum_is_call_s_km_3m is not null or sum_is_call_s_km_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0))
),

third_not_going_self as (
select epk_id, report_dt, '3' as target from arnsdpsbx_team_ss.bpm_multiclass_premier_main_feature_target_2025_07_31
where count_sale_vsp_3m is null and count_sale_sbol_3m is not null
and count_sales_with_predlozh_3m is null and sale_flag_erkc_3m is null
and (sum_is_call_s_dkm_3m is null or sum_is_call_s_dkm_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
and (sum_is_meet_km_3m is null or sum_is_meet_km_3m = 0)
and (sum_is_call_s_km_3m is null or sum_is_call_s_km_3m = 0)
and (sum_is_call_dkm_3m is null or sum_is_call_dkm_3m = 0)
),

four_not_going_km as (
select epk_id, report_dt, '4' as target from arnsdpsbx_team_ss.bpm_multiclass_premier_main_feature_target_2025_07_31
where count_sale_vsp_3m is null and count_sale_sbol_3m is null
and (count_sales_with_predlozh_3m is not null or sale_flag_erkc_3m is not null
or (sum_is_call_s_dkm_3m is not null or sum_is_call_s_dkm_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0)
or (sum_is_meet_km_3m is not null or sum_is_meet_km_3m > 0)
or (sum_is_call_s_km_3m is not null or sum_is_call_s_km_3m > 0)
or (sum_is_call_dkm_3m is not null or sum_is_call_dkm_3m > 0))
)

(select * from first_going_self)
union all
(select * from second_going_km)
union all
(select * from third_not_going_self)
union all
(select * from four_not_going_km)
''')

target_oot.write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_target_multiclass_oot', mode='overwrite')

In [7]:
spark.sql('''
select target, count(epk_id) from arnsdpsbx_team_ss.bpm_premier_target_multiclass_oot
group by target
order by target
''').show()

+------+-------------+
|target|count(epk_id)|
+------+-------------+
|     1|       182370|
|     2|       126205|
|     3|       564016|
|     4|       565277|
+------+-------------+

